# synent-task5-salesanalysis-rudrapatel
## Task 5: Sales Data Analysis - Superstore Sales Performance Analysis
**Prepared by:** Rudra Patel (Data Science Intern)  
**Company:** Synent Technologies  
**Date:** June 2, 2026  

---

### 📌 Introduction & Problem Statement
Analyzing large transactional sales databases is essential for identifying operational inefficiencies, seasonality, high-value product lines, and regional profit margins. Without proper analytics, businesses face profitability leaks due to sub-optimal pricing, excessive discounting, and inventory over-allocation.

### 🎯 Objectives
1. **Preprocess and clean** the transactional dataset.
2. **Visualize overall distributions** of Sales and Profit.
3. **Track monthly and annual revenue trajectories** to identify seasonal peaks.
4. **Analyze profitability margins** across categories, sub-categories, customer segments, and regions.
5. **Identify top product performers** and locate loss-making units.
6. **Provide data-backed business recommendations** to stabilize profit margins.

### 1. Environment Setup & Library Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

%matplotlib inline
sns.set_theme(style="whitegrid")

### 2. Dataset Loading & Initial Assessment

In [ ]:
raw_path = "../data/raw/Sample - Superstore.csv"
df = pd.read_csv(raw_path, encoding="latin-1")
print(f"Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nMissing values count per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

### 3. Data Cleaning & Preprocessing

In [ ]:
# 1. Drop duplicates
df_cleaned = df.drop_duplicates()

# 2. Convert date strings to datetime objects
df_cleaned['Order Date'] = pd.to_datetime(df_cleaned['Order Date'], format='%m/%d/%Y')
df_cleaned['Ship Date'] = pd.to_datetime(df_cleaned['Ship Date'], format='%m/%d/%Y')

# 3. Standardize Postal Code as a 5-digit string
df_cleaned['Postal Code'] = df_cleaned['Postal Code'].fillna(0).astype(int).astype(str).str.zfill(5)

# 4. Validate numerical bounds (ensuring Sales and Quantity are strictly positive)
df_cleaned = df_cleaned[df_cleaned['Sales'] > 0]
df_cleaned = df_cleaned[df_cleaned['Quantity'] > 0]

# Save cleaned output
os.makedirs("../data/processed", exist_ok=True)
df_cleaned.to_csv("../data/processed/cleaned_superstore.csv", index=False)
print(f"Cleaned dataset shape: {df_cleaned.shape}")

### 4. Exploratory Data Analysis (EDA) - Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Sales distribution (Log Scale)
sns.histplot(df_cleaned['Sales'], bins=50, kde=True, ax=axes[0], color='#1E3A8A', log_scale=True)
axes[0].set_title('Overall Sales Distribution (Log Scale)', fontweight='bold')
axes[0].set_xlabel('Sales ($)')

# Profit distribution (Zoomed range)
sns.histplot(df_cleaned[df_cleaned['Profit'].between(-200, 200)]['Profit'], bins=50, kde=True, ax=axes[1], color='#10B981')
axes[1].set_title('Profit Distribution (Concentrated -$200 to $200)', fontweight='bold')
axes[1].set_xlabel('Profit ($)')

plt.savefig("../images/sales_profit_distributions.png", dpi=300)
plt.show()

### 5. Monthly Revenue Analysis

In [ ]:
df_cleaned['YearMonth'] = df_cleaned['Order Date'].dt.to_period('M')
monthly_sales = df_cleaned.groupby('YearMonth')['Sales'].sum().reset_index()
monthly_sales['YearMonth_Str'] = monthly_sales['YearMonth'].astype(str)

plt.figure(figsize=(15, 6))
sns.lineplot(data=monthly_sales, x='YearMonth_Str', y='Sales', marker='o', linewidth=2.5, color='#1E3A8A')
plt.title('Monthly Sales Revenue Trends', fontsize=14, fontweight='bold')
plt.xlabel('Order Period (Year-Month)')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("../images/revenue_trends.png", dpi=300)
plt.show()

### 6. Profit & Margin Analysis

In [ ]:
# Category profitability margin
cat_profit = df_cleaned.groupby('Category')[['Sales', 'Profit']].sum().reset_index()
cat_profit['Margin_%'] = (cat_profit['Profit'] / cat_profit['Sales']) * 100
print("Category Margins:")
print(cat_profit)

# Sub-Category profit margins
subcat_profit = df_cleaned.groupby('Sub-Category')[['Sales', 'Profit']].sum().reset_index()
subcat_profit['Margin_%'] = (subcat_profit['Profit'] / subcat_profit['Sales']) * 100
subcat_sorted = subcat_profit.sort_values('Margin_%', ascending=False)

plt.figure(figsize=(12, 6))
colors = ['#EF4444' if x < 0 else '#10B981' for x in subcat_sorted['Margin_%']]
sns.barplot(data=subcat_sorted, y='Sub-Category', x='Margin_%', palette=colors)
plt.title('Sub-Category Profit Margins (%)', fontsize=14, fontweight='bold')
plt.xlabel('Profit Margin (%)')
plt.tight_layout()
plt.savefig("../images/profitability_analysis.png", dpi=300)
plt.show()

### 7. Product Performance Analysis

In [ ]:
top_sales_prod = df_cleaned.groupby('Product Name')['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_sales_prod, x='Sales', y='Product Name', palette='Blues_r')
plt.title('Top 10 Products by Sales Revenue ($)', fontsize=14, fontweight='bold')
plt.xlabel('Total Sales ($)')

# Shorten labels
ax = plt.gca()
ax.set_yticklabels([l.get_text()[:40]+'...' if len(l.get_text())>40 else l.get_text() for l in ax.get_yticklabels()])
plt.tight_layout()
plt.savefig("../images/product_analysis.png", dpi=300)
plt.show()

### 8. Regional & State-Level Performance Analysis

In [ ]:
state_perf = df_cleaned.groupby('State')[['Sales', 'Profit']].sum().reset_index()
state_perf['Margin_%'] = (state_perf['Profit'] / state_perf['Sales']) * 100
top_states = state_perf.sort_values(by='Sales', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_states, x='Sales', y='State', palette='Purples_r')
plt.title('Top 10 States by Sales Revenue ($)', fontsize=14, fontweight='bold')
plt.xlabel('Total Sales ($)')
plt.tight_layout()
plt.savefig("../images/regional_performance.png", dpi=300)
plt.show()

### 📌 Conclusions & Strategic Recommendations

#### Key Insights:
1. **Furniture category has a critical margin issue:** While generating high sales, it contributes only $2.49\%$ net margin due to low price control and heavy transport overheads.
2. **Discount leakage in states like Texas and Ohio:** Despite being in the top 10 states for sales, Texas and Ohio have substantial net losses due to excessive sales discounts.
3. **Technology & Office Supplies drive high profits:** Copiers, Phones, Paper, and Labels are the most profitable items and should be prioritized.

#### Recommendations:
- **Adjust Furniture Pricing:** Review the pricing structure of Tables and Bookcases; consider reducing stock of Tables due to a net loss of **-$17,725.48**.
- **Implement Regional Discount Controls:** Limit retail discounts to a maximum of $15\%$ in Texas and Ohio to recover lost margin.